# Klasifikasi Penyakit Daun Tomat — Modified K-Nearest Neighbor (MKNN)
### Fitur: HSV Color Moment + Local Binary Pattern (LBP)
### Split Data: Train 70% / Validation 15% / Test 15%

Notebook ini dijalankan di **Google Colab**. Alurnya:

1. Mount Google Drive (dataset `TomatoPlants` sudah ada di Drive)
2. **Split dataset menjadi Train : Val : Test = 70 : 15 : 15** (stratified per kelas) dan
   **menyalin fisik** hasil split ke folder baru di Drive (`TomatoPlants_Split/train`, `/val`, `/test`)
3. Preprocessing citra (resize)
4. Ekstraksi fitur warna HSV Color Moment (9 fitur) dan fitur tekstur LBP (5 fitur)
5. Implementasi algoritma **Modified K-Nearest Neighbor** (validity + weighted voting)
6. **Pencarian nilai K secara otomatis** — dipilih berdasarkan akurasi tertinggi pada **data Validation**
7. **Evaluasi akhir pada data Test** (data yang sama sekali tidak dipakai untuk memilih K, sehingga hasil evaluasi tidak bias)
8. Menyimpan model (`model_mknn.pkl`) untuk dipakai oleh backend Flask

**Struktur dataset awal di Google Drive:**
```
MyDrive/
└── TomatoPlants/
    ├── Bercak Kering/            (1000 gambar)
    ├── Busuk Daun Fitoftora/     (1000 gambar)
    └── Sehat/                    (1000 gambar)
```

**Struktur folder hasil split yang akan dibuat di Drive:**
```
MyDrive/
└── TomatoPlants_Split/
    ├── train/
    │   ├── Bercak Kering/            (700 gambar)
    │   ├── Busuk Daun Fitoftora/     (700 gambar)
    │   └── Sehat/                    (700 gambar)
    ├── val/
    │   ├── Bercak Kering/            (150 gambar)
    │   ├── Busuk Daun Fitoftora/     (150 gambar)
    │   └── Sehat/                    (150 gambar)
    └── test/
        ├── Bercak Kering/            (150 gambar)
        ├── Busuk Daun Fitoftora/     (150 gambar)
        └── Sehat/                    (150 gambar)
```


## 1. Instalasi Library & Import

In [ ]:
!pip install -q scikit-image scikit-learn opencv-python-headless

import os
import glob
import shutil
import pickle
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import skew
from skimage.feature import local_binary_pattern
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

np.random.seed(42)


## 2. Mount Google Drive & Konfigurasi Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Sesuaikan path ini apabila lokasi folder TomatoPlants berbeda di Drive Anda
DATASET_PATH = '/content/drive/MyDrive/TomatoPlants'
SPLIT_PATH = '/content/drive/MyDrive/TomatoPlants_Split'

CLASS_NAMES = ['Bercak Kering', 'Busuk Daun Fitoftora', 'Sehat']
IMG_SIZE = 128  # resize citra (preprocessing)

# Proporsi split data
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9

for c in CLASS_NAMES:
    folder = os.path.join(DATASET_PATH, c)
    jumlah = len(glob.glob(os.path.join(folder, '*')))
    print(f'{c:25s}: {jumlah} gambar')


## 3. Split Dataset (Train : Val : Test = 70 : 15 : 15) & Simpan ke Drive

Split dilakukan **per kelas (stratified)** agar proporsi tiap kelas tetap seimbang di
Train, Val, maupun Test. Hasil split kemudian **disalin fisik** (bukan hanya index) ke
folder baru `TomatoPlants_Split/` di Google Drive, supaya:
- Anda punya arsip dataset yang sudah terbagi rapi untuk lampiran laporan/skripsi
- Proses training bisa diulang tanpa harus split ulang dari awal


In [ ]:
def split_and_copy_dataset():
    split_summary = []

    for split_name in ['train', 'val', 'test']:
        for c in CLASS_NAMES:
            os.makedirs(os.path.join(SPLIT_PATH, split_name, c), exist_ok=True)

    for c in CLASS_NAMES:
        folder = os.path.join(DATASET_PATH, c)
        files = sorted(glob.glob(os.path.join(folder, '*.*')))

        # Split pertama: train vs (val + test)
        train_files, temp_files = train_test_split(
            files, train_size=TRAIN_RATIO, random_state=42, shuffle=True
        )
        # Split kedua: val vs test, dari sisa temp_files
        relative_test_ratio = TEST_RATIO / (VAL_RATIO + TEST_RATIO)
        val_files, test_files = train_test_split(
            temp_files, test_size=relative_test_ratio, random_state=42, shuffle=True
        )

        for split_name, file_list in [('train', train_files), ('val', val_files), ('test', test_files)]:
            dest_dir = os.path.join(SPLIT_PATH, split_name, c)
            for f in file_list:
                shutil.copy2(f, os.path.join(dest_dir, os.path.basename(f)))

        split_summary.append({
            'Kelas': c,
            'Train': len(train_files),
            'Val': len(val_files),
            'Test': len(test_files),
            'Total': len(files),
        })

    return split_summary


split_summary = split_and_copy_dataset()

import pandas as pd
df_summary = pd.DataFrame(split_summary)
df_summary.loc['Total'] = df_summary.sum(numeric_only=True)
df_summary.loc['Total', 'Kelas'] = 'TOTAL'
print(df_summary.to_string(index=False))
print('\nFolder hasil split tersimpan di:', SPLIT_PATH)


## 4. Preprocessing & Ekstraksi Fitur

**Preprocessing**: resize citra ke `128x128`.

**Fitur warna — HSV Color Moment (9 fitur)**: mean, standar deviasi, skewness untuk tiap channel H, S, V.

**Fitur tekstur — Local Binary Pattern (5 fitur statistik orde pertama)**: mean, variance,
skewness, kurtosis, entropy dari histogram LBP citra grayscale.

Total fitur per citra = 9 (HSV) + 5 (LBP) = **14 fitur**.


In [ ]:
def extract_hsv_color_moment(img_bgr):
    """Menghasilkan 9 fitur warna: mean, std, skewness untuk tiap channel H, S, V"""
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    feats = []
    for ch in range(3):
        channel = hsv[:, :, ch].astype(np.float64).flatten()
        mean_val = np.mean(channel)
        std_val = np.std(channel)
        skew_val = skew(channel)
        feats.extend([mean_val, std_val, skew_val])
    return feats


def extract_lbp_features(img_bgr, P=8, R=1):
    """Menghasilkan 5 fitur tekstur orde pertama dari histogram LBP:
    mean, variance, skewness, kurtosis, entropy"""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    lbp = local_binary_pattern(gray, P=P, R=R, method='default')

    hist, _ = np.histogram(lbp.ravel(), bins=256, range=(0, 256))
    hist = hist.astype(np.float64)
    prob = hist / (hist.sum() + 1e-9)

    idx = np.arange(256)
    mean_val = np.sum(idx * prob)
    variance = np.sum(((idx - mean_val) ** 2) * prob)
    std_val = np.sqrt(variance) if variance > 0 else 1e-9
    skewness = np.sum(((idx - mean_val) ** 3) * prob) / (std_val ** 3 + 1e-9)
    kurtosis_val = (np.sum(((idx - mean_val) ** 4) * prob) / (std_val ** 4 + 1e-9)) - 3
    nz = prob[prob > 0]
    entropy_val = -np.sum(nz * np.log2(nz))

    return [mean_val, variance, skewness, kurtosis_val, entropy_val]


def preprocess_image(path, img_size=IMG_SIZE):
    img = cv2.imread(path)
    if img is None:
        raise ValueError(f'Gagal membaca gambar: {path}')
    img = cv2.resize(img, (img_size, img_size))
    return img


def extract_features(path):
    img = preprocess_image(path)
    hsv_feats = extract_hsv_color_moment(img)
    lbp_feats = extract_lbp_features(img)
    return hsv_feats + lbp_feats


def load_split_features(split_name):
    """Ekstraksi fitur untuk seluruh gambar pada satu folder split (train/val/test)"""
    X, y = [], []
    for c in CLASS_NAMES:
        folder = os.path.join(SPLIT_PATH, split_name, c)
        files = sorted(glob.glob(os.path.join(folder, '*.*')))
        for f in files:
            try:
                X.append(extract_features(f))
                y.append(c)
            except Exception as e:
                print(f'  Lewati {f}: {e}')
    return np.array(X), np.array(y)


## 5. Ekstraksi Fitur untuk Train, Val, dan Test

In [ ]:
print('Mengekstrak fitur data TRAIN...')
X_train_raw, y_train = load_split_features('train')
print('  ->', X_train_raw.shape)

print('Mengekstrak fitur data VAL...')
X_val_raw, y_val = load_split_features('val')
print('  ->', X_val_raw.shape)

print('Mengekstrak fitur data TEST...')
X_test_raw, y_test = load_split_features('test')
print('  ->', X_test_raw.shape)


## 6. Normalisasi Fitur

Scaler (`MinMaxScaler`) di-*fit* **hanya pada data Train**, lalu dipakai untuk mentransformasi
data Val dan Test. Ini penting agar tidak terjadi *data leakage* (informasi dari Val/Test
"bocor" ke proses normalisasi).


In [ ]:
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val = scaler.transform(X_val_raw)
X_test = scaler.transform(X_test_raw)

print('Data latih   (train) :', X_train.shape[0])
print('Data validasi (val)  :', X_val.shape[0])
print('Data uji     (test)  :', X_test.shape[0])


## 7. Implementasi Algoritma Modified K-Nearest Neighbor (MKNN)

Tahapan MKNN mengikuti proposal:

1. **Jarak Euclidean** antar data (Persamaan 2.6)
2. **Nilai Validitas** tiap data latih berdasarkan H tetangga terdekatnya (Persamaan 2.7–2.8)
3. **Weighted Voting**: `W(i) = Validity(i) * 1/(de + 0.5)` (Persamaan 2.9)
4. Kelas dengan total weighted vote tertinggi menjadi hasil klasifikasi


In [ ]:
class ModifiedKNN:
    def __init__(self, k=5, H=5, a=0.5):
        self.k = k
        self.H = H
        self.a = a

    def fit(self, X, y):
        self.X_train = np.asarray(X)
        self.y_train = np.asarray(y)
        self.validity_ = self._compute_validity()
        return self

    @staticmethod
    def _euclidean(X, x):
        return np.sqrt(np.sum((X - x) ** 2, axis=1))

    def _compute_validity(self):
        n = len(self.X_train)
        validity = np.zeros(n)
        for i in range(n):
            dists = self._euclidean(self.X_train, self.X_train[i])
            nn_idx = np.argsort(dists)[1:self.H + 1]
            same_label = np.sum(self.y_train[nn_idx] == self.y_train[i])
            validity[i] = same_label / self.H
        return validity

    def predict(self, X_test):
        preds = []
        for x in np.asarray(X_test):
            dists = self._euclidean(self.X_train, x)
            k_idx = np.argsort(dists)[: self.k]
            weight_per_class = {}
            for i in k_idx:
                w = self.validity_[i] * (1.0 / (dists[i] + self.a))
                label = self.y_train[i]
                weight_per_class[label] = weight_per_class.get(label, 0.0) + w
            pred_label = max(weight_per_class, key=weight_per_class.get)
            preds.append(pred_label)
        return np.array(preds)


## 8. Menentukan Nilai K Secara Otomatis (berdasarkan data Validation)

Nilai K **tidak ditentukan manual**. Sistem mencoba beberapa kandidat nilai K (ganjil, 1 sampai 25),
melatih MKNN pada data **Train**, lalu mengukur akurasi terhadap data **Validation**. Nilai K dengan
akurasi validasi tertinggi dipilih secara otomatis sebagai K final (`best_k`).

Data **Test** sama sekali tidak disentuh pada tahap ini — baru dipakai di evaluasi akhir (Bab 9).


In [ ]:
H_NEIGHBORS = 5  # jumlah tetangga untuk menghitung nilai validitas
k_candidates = list(range(1, 26, 2))
val_accuracies = []

best_k, best_acc, best_model = None, -1, None

for k in k_candidates:
    model = ModifiedKNN(k=k, H=H_NEIGHBORS)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    acc = accuracy_score(y_val, y_pred)
    val_accuracies.append(acc)
    print(f'k = {k:2d}  ->  akurasi validasi = {acc:.4f}')
    if acc > best_acc:
        best_acc = acc
        best_k = k
        best_model = model

print(f'\n>>> Nilai K terbaik dipilih otomatis (berdasarkan data VALIDATION): K = {best_k} (akurasi validasi = {best_acc:.4f})')


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(k_candidates, val_accuracies, marker='o')
plt.axvline(best_k, color='red', linestyle='--', label=f'K terbaik = {best_k}')
plt.title('Pencarian Nilai K Otomatis (MKNN) - Akurasi Validasi')
plt.xlabel('Nilai K')
plt.ylabel('Akurasi Validasi')
plt.legend()
plt.grid(True)
plt.show()


## 9. Evaluasi Akhir pada Data TEST (Unseen Data)

Ini adalah evaluasi performa model yang sesungguhnya, karena data Test **tidak pernah**
dipakai baik untuk melatih model maupun untuk memilih nilai K.


In [ ]:
y_pred_test = best_model.predict(X_test)

acc = accuracy_score(y_test, y_pred_test)
prec = precision_score(y_test, y_pred_test, average='macro')
rec = recall_score(y_test, y_pred_test, average='macro')
f1 = f1_score(y_test, y_pred_test, average='macro')

print(f'K optimal (dipilih dari data Val) : {best_k}')
print(f'Akurasi (Test)            : {acc:.4f}')
print(f'Precision (Test, macro)   : {prec:.4f}')
print(f'Recall (Test, macro)      : {rec:.4f}')
print(f'F1-Score (Test, macro)    : {f1:.4f}')
print()
print(classification_report(y_test, y_pred_test))


In [ ]:
cm = confusion_matrix(y_test, y_pred_test, labels=CLASS_NAMES)

plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title(f'Confusion Matrix pada Data TEST — MKNN (K={best_k})')
plt.xlabel('Prediksi')
plt.ylabel('Aktual')
plt.tight_layout()
plt.show()


## 10. Menyimpan Model untuk Backend Flask

Model final dilatih di atas **data Train** (bukan Train+Val), sesuai kaidah split 3 arah:
Train untuk melatih, Val untuk memilih K, Test untuk melaporkan performa akhir yang tidak bias.

Karena MKNN adalah *lazy learner*, seluruh data latih beserta nilai validitasnya perlu ikut
disimpan. File `model_mknn.pkl` ini nantinya diletakkan di folder `backend/model/` pada
project Flask.


In [ ]:
model_artifact = {
    'X_train': best_model.X_train,
    'y_train': best_model.y_train,
    'validity': best_model.validity_,
    'best_k': best_k,
    'H': H_NEIGHBORS,
    'a': best_model.a,
    'scaler': scaler,
    'class_names': CLASS_NAMES,
    'img_size': IMG_SIZE,
    # metrik evaluasi akhir (test set) ikut disimpan sebagai dokumentasi
    'test_accuracy': float(acc),
    'test_precision_macro': float(prec),
    'test_recall_macro': float(rec),
    'test_f1_macro': float(f1),
}

with open('model_mknn.pkl', 'wb') as f:
    pickle.dump(model_artifact, f)

print('Model tersimpan sebagai model_mknn.pkl')
print('Ukuran file:', os.path.getsize('model_mknn.pkl') / 1024, 'KB')


In [ ]:
# (Opsional) unduh langsung ke komputer lokal
from google.colab import files
files.download('model_mknn.pkl')

# (Opsional) atau simpan ke Google Drive agar mudah diambil nanti
# shutil.copy('model_mknn.pkl', '/content/drive/MyDrive/model_mknn.pkl')


---
### Ringkasan

- Dataset di-split **Train 70% : Val 15% : Test 15%** (stratified per kelas) dan hasil split
  disalin fisik ke `MyDrive/TomatoPlants_Split/{train,val,test}/`.
- Nilai **K dipilih otomatis** berdasarkan akurasi data **Validation**.
- Performa akhir (akurasi, precision, recall, F1, confusion matrix) dilaporkan dari data
  **Test** yang sama sekali tidak dipakai saat training maupun pemilihan K.

### Langkah selanjutnya
1. Unduh `model_mknn.pkl` dari Colab (cell di atas otomatis memicu unduhan).
2. Salin file tersebut ke folder **`backend/model/model_mknn.pkl`** pada project Flask.
3. Jalankan backend Flask seperti biasa — tidak ada perubahan pada sisi Flask, karena struktur
   `model_mknn.pkl` (X_train, y_train, validity, scaler, best_k, dst.) tetap sama.
